# Topic Modeling with BERTopic

Advancing AI Anthropology through computational approaches to qualitative research.

[Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

<br>

---

<br>

## What This Notebook Does

This notebook discovers topics in text collections using BERTopic, a transformer-based topic modeling framework. Upload documents (interview transcripts, field notes, articles, or any text corpus), and the notebook clusters them by semantic similarity, identifies topics, and produces interactive visualizations showing how topics relate to each other.

Unlike traditional topic modeling (LDA), BERTopic uses transformer embeddings to understand meaning, not just word frequency. This produces more coherent and interpretable topics, especially for qualitative research data where context matters.

## Key Features

1. **Transformer-Based**: Uses sentence embeddings for semantically meaningful topic clusters
2. **Zero-Shot Mode**: Optionally provide expected topics and let BERTopic find them in your data
3. **Multi-Format Input**: Upload CSV, TXT, DOCX, or PDF files, or paste text directly
4. **Interactive Visualizations**: Topic maps, hierarchy trees, bar charts, and similarity heatmaps
5. **Topic Search**: Find topics similar to any keyword or phrase
6. **Configurable Parameters**: Control topic count, minimum topic size, and n-gram range
7. **Structured Export**: Download topic assignments, topic summaries, and representative documents

## Workflow

1. **Upload**: Load your text corpus from files or paste directly
2. **Configure**: Set parameters and optionally define expected topics (zero-shot mode)
3. **Model**: Run BERTopic to discover topics in your data
4. **Explore**: Review visualizations and topic details
5. **Export**: Download topic assignments and summaries for further analysis

## Applications

This notebook supports research that involves discovering thematic structure in text collections. It complements the Coding and Thematic Analysis notebook (which applies predefined codes) by offering unsupervised topic discovery. Useful for exploratory analysis of interview corpora, literature collections, media coverage, field notes, and any text dataset where you want to identify emergent themes.

## Methodological Positioning

This notebook represents a **computational anthropology** approach. BERTopic discovers statistical patterns in text, not cultural meaning. The topics it identifies are starting points for interpretation, not conclusions. The researcher's ethnographic knowledge remains essential for evaluating whether discovered topics are analytically meaningful.

**Important**: Topic modeling is exploratory. Results depend on corpus composition, preprocessing choices, and parameter settings. Run with different configurations and compare results before drawing conclusions.

## Target Audience

Designed for anthropologists and qualitative researchers who want to explore thematic structure in text collections before or alongside manual coding.

## Technical Approach

The notebook uses **BERTopic** with sentence-transformer embeddings, UMAP dimensionality reduction, and HDBSCAN clustering. Topic labels are generated using KeyBERTInspired representation (no API key required). Zero-shot mode uses cosine similarity to match documents to predefined topics. All processing runs in the notebook.

## License & Attribution

This notebook is released under the **CC BY-NC 4.0** license. You may adapt and share it for non-commercial purposes with proper attribution.

## Citation

If you use this toolkit in your academic research, please cite:

> Artz, Matt. 2025. AI Anthropology Toolkit. Software. Zenodo. https://doi.org/10.5281/zenodo.16728812

## References

Artz, Matt. 2023. From Machine Learning to Machine Knowing: A Digital Anthropology Approach for the Machine Interpretation of Cultures. UNESCO. https://unesdoc.unesco.org/ark:/48223/pf0000384902.

Artz, Matt. 2023. "Ten Predictions for AI and the Future of Anthropology." Anthropology News, May 8. https://doi.org/10.1111/AN.1605.

Artz, Matt. 2026. "Artificial Intelligence: The AI Anthropology Lifecycle (of, by, for AI)." In Practicing Digital Ethnography, edited by Devin Proctor. Routledge. https://doi.org/10.4324/9781032672663-29.

Artz, Matt. 2026. "Multi-Agent Ethnography: Post-Conventional Anthropological Practice Through Human-AI Collaboration." Human Organization. https://doi.org/10.1080/00664677.2026.2614501.

Artz, Matt. Forthcoming. "AI Anthropology: The Future of Applied Anthropological Practice." In Routledge Handbook of Applied Anthropology, edited by Christina Wasson, Edward B. Liebow, Karine L. Narahara, Ndukuyakhe Ndlovu, and Alaka Wali. New York: Routledge.

Grootendorst, Maarten. 2022. "BERTopic: Neural Topic Modeling with a Class-Based TF-IDF Procedure." arXiv preprint arXiv:2203.05794.

## Setup and Installation

Install required packages and import libraries. This cell installs BERTopic and its dependencies. In Google Colab, a GPU runtime is recommended for faster processing (Runtime > Change runtime type > T4 GPU).

In [ ]:
# Install required packages
!pip install bertopic pandas openpyxl ipywidgets python-docx nltk pypdf -q

import re
import unicodedata
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
logging.getLogger("transformers.configuration_utils").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("safetensors").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

# Academic boilerplate terms that don't distinguish topics in scholarly corpora
ACADEMIC_STOPWORDS = STOP_WORDS | {
    'study', 'research', 'analysis', 'approach', 'paper', 'article', 'findings',
    'results', 'method', 'methods', 'methodology', 'literature', 'data', 'framework',
    'theory', 'theoretical', 'concept', 'conceptual', 'argue', 'argument', 'argues',
    'suggest', 'suggests', 'examine', 'examines', 'explore', 'explores', 'discuss',
    'discusses', 'demonstrate', 'demonstrates', 'provide', 'provides', 'note', 'notes',
    'describe', 'describes', 'explain', 'explains', 'consider', 'considers',
    'introduction', 'conclusion', 'abstract', 'chapter', 'section', 'volume',
    'journal', 'press', 'university', 'review', 'cited', 'reference', 'references',
    'scholar', 'scholars', 'scholarly', 'academic', 'author', 'authors', 'edited',
    'editor', 'editors', 'published', 'publication', 'pp', 'vol', 'eds', 'doi',
    'isbn', 'issn', 'http', 'https', 'www', 'org', 'com', 'pdf',
    'york', 'london', 'cambridge', 'oxford', 'routledge', 'springer', 'wiley',
    'forthcoming', 'preprint', 'arxiv', 'zenodo',
    'also', 'however', 'although', 'within', 'across', 'among', 'between',
    'rather', 'particular', 'particularly', 'specific', 'specifically',
    'important', 'significant', 'new', 'different', 'various', 'many',
    'work', 'working', 'way', 'ways', 'form', 'forms', 'process', 'role',
    'based', 'context', 'practice', 'practices', 'example', 'case',
    'understanding', 'perspective', 'perspectives', 'question', 'questions',
}

from sklearn.feature_extraction.text import CountVectorizer


def preprocess_text(text):
    """Preprocess text for topic modeling."""
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(w) for w in tokens if w not in STOP_WORDS and len(w) > 2]
    return ' '.join(tokens)


def normalize_text(text):
    """Normalize unicode characters."""
    if not isinstance(text, str): return text
    text = unicodedata.normalize('NFKC', text)
    for old, new in [('\u2011','-'),('\u2013','-'),('\u2014','-'),('\u2018',"'"),('\u2019',"'"),('\u201c','"'),('\u201d','"'),('\u2026','...')]:
        text = text.replace(old, new)
    return text


def make_slug(query):
    """Create a clean filename slug."""
    return re.sub(r'[^a-zA-Z0-9]+', '_', query).strip('_')[:30] or 'topics'


def style_excel(filepath):
    """Apply branded styling to an Excel file."""
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    wb = load_workbook(filepath)
    hf = PatternFill(start_color='274C77', end_color='274C77', fill_type='solid')
    hfont = Font(bold=True, color='FFFFFF')
    tb = Border(left=Side(style='thin',color='E7ECEF'), right=Side(style='thin',color='E7ECEF'),
               top=Side(style='thin',color='E7ECEF'), bottom=Side(style='thin',color='E7ECEF'))
    for ws in wb.worksheets:
        for cell in ws[1]: cell.fill, cell.font, cell.alignment, cell.border = hf, hfont, Alignment(horizontal='center', vertical='center', wrap_text=True), tb
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = min(max(max(len(str(c.value or '')) for c in col)+2, 10), 60)
        ws.freeze_panes = 'A2'
        for row in ws.iter_rows(min_row=2):
            for cell in row: cell.alignment, cell.border = Alignment(vertical='top', wrap_text=True), tb
    wb.save(filepath)


print("\u2713 All packages installed and libraries loaded successfully")
env_name = "Google Colab" if IN_COLAB else "Local Jupyter"
print(f"\u2713 Environment: {env_name}")
print("\U0001f4ca Ready to configure topic modeling!")

## Data Input

Upload your text data or paste it directly. Supports CSV, TXT, DOCX, and PDF files. Reference sections are automatically stripped and short paragraphs are merged into substantial text segments for better topic discovery.

In [ ]:
# Data Input Interface

MIN_CHUNK_WORDS = 150  # Merge short paragraphs into substantial chunks for better embeddings

def strip_references_section(paragraphs):
    """Remove bibliography/references section from the end of a document.
    Detects common heading patterns and drops everything after them."""
    ref_headings = {'references', 'bibliography', 'works cited', 'literature cited',
                    'sources cited', 'endnotes', 'notes and references'}
    cutoff = len(paragraphs)
    for i, para in enumerate(paragraphs):
        stripped = para.strip().lower().rstrip(':').rstrip('.')
        # Match "References" as a standalone heading (short line, matches a known heading)
        if stripped in ref_headings and len(para.split()) <= 4:
            cutoff = i
            break
    return paragraphs[:cutoff]


def clean_citation_noise(text):
    """Remove inline citation patterns that pollute topic keywords."""
    # Remove patterns like (Author 2024), (Author & Author, 2023), (Author et al. 2025)
    text = re.sub(r'\([A-Z][a-z]+(?:\s+(?:and|&)\s+[A-Z][a-z]+)*(?:\s+et\s+al\.?)?[,\s]+(?:19|20)\d{2}[a-z]?\)', '', text)
    # Remove "Author (2024)" patterns
    text = re.sub(r'[A-Z][a-z]+\s+\((?:19|20)\d{2}[a-z]?\)', '', text)
    # Remove standalone years in citation contexts
    text = re.sub(r'\b(?:19|20)\d{2}[a-z]?\b', '', text)
    # Remove DOIs
    text = re.sub(r'https?://doi\.org/[^\s]+', '', text)
    text = re.sub(r'doi:\s*[^\s]+', '', text)
    return text


def is_junk_segment(text):
    """Detect segments that are mostly numeric, tables, or non-prose content."""
    words = text.split()
    if len(words) < 10:
        return True
    # Count alphabetic vs non-alphabetic tokens
    alpha_count = sum(1 for w in words if re.match(r'^[a-zA-Z]{2,}', w))
    alpha_ratio = alpha_count / len(words)
    return alpha_ratio < 0.5

style = {'description_width': '120px'}
layout = widgets.Layout(width='500px')

input_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 20px 0; border-left: 5px solid #274C77;">'
input_html += '<h2 style="color: #274C77; margin-top: 0;">\U0001f4ca Topic Modeling with BERTopic</h2>'
input_html += '<p><strong>Welcome!</strong> This notebook discovers topics in your text data using transformer-based clustering.</p>'
input_html += '<h3 style="color: #274C77;">\U0001f4d6 How to Use:</h3>'
input_html += '<ol>'
input_html += '<li><strong>Upload:</strong> Add your text data below (CSV, TXT, DOCX, or PDF)</li>'
input_html += '<li><strong>Configure:</strong> Set parameters in the next cell</li>'
input_html += '<li><strong>Run:</strong> Execute the model to discover topics</li>'
input_html += '<li><strong>Explore:</strong> Review visualizations and export results</li>'
input_html += '</ol>'
input_html += '</div>'

input_instructions = widgets.HTML(value=input_html)

input_method = widgets.ToggleButtons(
    options=[('Upload Files', 'upload'), ('Paste Text', 'paste')],
    value='upload',
    style={'button_width': '150px'}
)

# CSV column setting (shown in upload mode)
csv_col = widgets.Text(value='text', description='Text column:', placeholder='Column name containing text (for CSV)', layout=layout, style=style)
csv_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">For CSV files, specify which column contains the text to analyze.</p>')

# Paste widget
paste_area = widgets.Textarea(value='', placeholder='Paste documents here, one per line (or separated by blank lines)', layout=widgets.Layout(width='100%', height='200px'))

# Storage
documents = []
doc_labels = []
upload_out = widgets.Output()

upload_box = widgets.VBox([csv_col, csv_help])
paste_box = widgets.VBox([paste_area])
paste_box.layout.display = 'none'

def on_method_change(change):
    if change['new'] == 'upload':
        upload_box.layout.display = ''
        paste_box.layout.display = 'none'
    else:
        upload_box.layout.display = 'none'
        paste_box.layout.display = ''

input_method.observe(on_method_change, names='value')


def merge_short_paragraphs(paragraphs, min_words=MIN_CHUNK_WORDS):
    """Merge consecutive short paragraphs into chunks of at least min_words.

    Prevents BERTopic from getting tiny fragments like headings and captions
    as standalone documents. Adjacent paragraphs are combined until the chunk
    reaches the minimum word count.
    """
    chunks = []
    current = []
    current_words = 0

    for para in paragraphs:
        words = len(para.split())
        current.append(para)
        current_words += words

        if current_words >= min_words:
            chunks.append(' '.join(current))
            current = []
            current_words = 0

    # Append remainder
    if current:
        if chunks:
            # Merge trailing short fragment into the last chunk
            chunks[-1] += ' ' + ' '.join(current)
        else:
            chunks.append(' '.join(current))

    return chunks


load_btn = widgets.Button(description='Load Documents', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='20px 0'))

def on_load_clicked(_):
    global documents, doc_labels
    upload_out.clear_output()
    documents = []
    doc_labels = []

    with upload_out:
        if input_method.value == 'paste':
            raw = paste_area.value.strip()
            if not raw:
                print("\u26a0\ufe0f Please paste some text.")
                return
            # Split by blank lines or single lines
            docs = [d.strip() for d in re.split(r'\n\s*\n', raw) if d.strip()]
            if len(docs) == 1:
                docs = [d.strip() for d in raw.split('\n') if d.strip()]
            docs = merge_short_paragraphs(docs)
            docs = [d for d in docs if not is_junk_segment(d)]
            documents = docs
            doc_labels = [f'doc_{i+1}' for i in range(len(docs))]
            print(f"\u2713 Loaded {len(documents)} text segments from pasted text")

        elif input_method.value == 'upload':
            if IN_COLAB:
                from google.colab import files as colab_files
                uploaded = colab_files.upload()
                if not uploaded:
                    print("\u26a0\ufe0f No files uploaded.")
                    return
                for fname, content in uploaded.items():
                    if fname.endswith('.csv'):
                        import io
                        df = pd.read_csv(io.BytesIO(content))
                        col = csv_col.value.strip()
                        if col not in df.columns:
                            print(f"\u26a0\ufe0f Column '{col}' not found. Available: {list(df.columns)}")
                            return
                        docs = df[col].dropna().astype(str).tolist()
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.txt'):
                        text = content.decode('utf-8', errors='ignore')
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = merge_short_paragraphs(paras)
                        docs = [d for d in docs if not is_junk_segment(d)]
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.docx'):
                        import docx, io
                        doc = docx.Document(io.BytesIO(content))
                        paras = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = merge_short_paragraphs(paras)
                        docs = [d for d in docs if not is_junk_segment(d)]
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    elif fname.endswith('.pdf'):
                        import io
                        try:
                            from pypdf import PdfReader
                        except ImportError:
                            from PyPDF2 import PdfReader
                        reader = PdfReader(io.BytesIO(content))
                        text = '\n'.join(page.extract_text() or '' for page in reader.pages)
                        paras = [d.strip() for d in text.split('\n') if d.strip()]
                        paras = strip_references_section(paras)
                        paras = [clean_citation_noise(p) for p in paras]
                        docs = merge_short_paragraphs(paras)
                        docs = [d for d in docs if not is_junk_segment(d)]
                        documents.extend(docs)
                        doc_labels.extend([f'{fname}_{i+1}' for i in range(len(docs))])
                    else:
                        print(f"\u26a0\ufe0f Unsupported format: {fname}")
                        continue
                print(f"\u2713 Loaded {len(documents)} text segments from {len(uploaded)} file(s)")
            else:
                print("\u26a0\ufe0f File upload widget works differently locally. Use Paste Text mode, or load from CSV:")
                print("   Upload a CSV to your notebook directory and set the path below.")

        if documents:
            avg_len = np.mean([len(d.split()) for d in documents])
            print(f"   Average segment length: {avg_len:.0f} words")
            if len(documents) < 20:
                print(f"   \u26a0\ufe0f Small corpus ({len(documents)} segments). BERTopic works best with 50+ segments.")

load_btn.on_click(on_load_clicked)

display(widgets.VBox([
    input_instructions,
    input_method,
    upload_box,
    paste_box,
    load_btn,
    upload_out,
]))

## Model Configuration

Configure BERTopic parameters. In standard mode, BERTopic discovers topics automatically. In zero-shot mode, you provide a list of expected topics and BERTopic finds documents matching them.

In [ ]:
# Model Configuration

config_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\u2699\ufe0f Parameters</h3>')

mode_toggle = widgets.ToggleButtons(
    options=[('Standard (auto-discover)', 'standard'), ('Zero-Shot (guided)', 'zeroshot')],
    value='standard',
    style={'button_width': '200px'}
)

mode_help = widgets.HTML(
    '<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">'
    '<strong>Standard:</strong> BERTopic discovers topics automatically from your data.<br>'
    '<strong>Zero-Shot:</strong> Provide expected topics. Documents matching them are assigned; the rest are clustered normally.</p>'
)

min_topic_input = widgets.IntSlider(value=5, min=2, max=50, step=1, description='Min topic size:', style=style)
min_topic_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Minimum number of documents to form a topic. Lower = more topics, higher = fewer broader topics.</p>')

nr_topics_input = widgets.Dropdown(
    options=[('Auto', 'auto'), ('10', 10), ('20', 20), ('30', 30), ('50', 50), ('No reduction', None)],
    value='auto',
    description='Nr topics:',
    style=style
)

ngram_input = widgets.Dropdown(
    options=[('Unigrams (1)', 1), ('Bigrams (1-2)', 2), ('Trigrams (1-3)', 3)],
    value=2,
    description='N-gram range:',
    style=style
)

preprocess_chk = widgets.Checkbox(value=False, description='Preprocess text before embedding (usually not needed — leave off for best results)', indent=False, style={'description_width': 'auto'})

# Zero-shot config
zeroshot_header = widgets.HTML('<h3 style="margin: 20px 0 10px 0; color: #274C77;">\U0001f3af Zero-Shot Topics</h3>')
zeroshot_input = widgets.Textarea(
    value='',
    placeholder='Enter expected topics, one per line (e.g., "Healthcare access", "Community resilience")',
    layout=widgets.Layout(width='500px', height='100px')
)
zeroshot_sim = widgets.FloatSlider(value=0.85, min=0.5, max=0.99, step=0.01, description='Min similarity:', style=style)
zeroshot_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em; margin: 2px 0 10px 0;">Documents below this similarity threshold will be clustered normally instead of assigned to a zero-shot topic.</p>')

zeroshot_box = widgets.VBox([zeroshot_header, zeroshot_input, zeroshot_sim, zeroshot_help])
zeroshot_box.layout.display = 'none'

def on_mode_toggle(change):
    zeroshot_box.layout.display = '' if change['new'] == 'zeroshot' else 'none'
mode_toggle.observe(on_mode_toggle, names='value')

display(widgets.VBox([
    config_header,
    mode_toggle, mode_help,
    min_topic_input, min_topic_help,
    nr_topics_input, ngram_input,
    preprocess_chk,
    zeroshot_box,
]))

## Run Topic Model

Execute BERTopic on your loaded documents. This may take a few minutes depending on corpus size. Progress updates appear below.

In [ ]:
# Run BERTopic

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

topic_model = None
topics = None
probs = None
docs_processed = None
docs_original = None  # Keep originals for display
labels_valid = None

run_btn = widgets.Button(description='Run Topic Model', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
progress = widgets.HTML(value='')
run_out = widgets.Output()

def _word_stem(w):
    """Rough stemming for dedup — strip common suffixes."""
    w = w.lower()
    for suffix in ['ological', 'ology', 'ation', 'ment', 'ness', 'ical', 'ious', 'ous', 'ive', 'ing', 'ity', 'ist', 'ism', 'ies', 'ers', 'ent', 'ant', 'ble', 'al', 'ly', 'ed', 'er', 'es', 'ss']:
        if w.endswith(suffix) and len(w) - len(suffix) >= 3:
            return w[:len(w) - len(suffix)]
    return w


def clean_topic_label(name, topic_id=None, topic_model_ref=None):
    """Convert BERTopic's auto-label into a readable theme name.
    Filters corpus-wide terms and deduplicates morphological variants."""
    if not name or str(topic_id) == '-1' or str(name).startswith('-1'):
        return 'Outliers (unassigned)'

    if topic_model_ref and topic_id is not None:
        topic_words = topic_model_ref.get_topic(topic_id)
        if topic_words:
            all_words = [w for w, _ in topic_words[:15]]

            # Find corpus-wide terms by stem
            all_topic_info = topic_model_ref.get_topic_info()
            real_topics = [t for t in all_topic_info['Topic'] if t != -1]
            common_stems = set()
            if len(real_topics) > 1:
                stem_topic_count = {}
                for tid in real_topics:
                    tw = topic_model_ref.get_topic(tid)
                    if tw:
                        seen_in_topic = set()
                        for w, _ in tw[:10]:
                            stem = _word_stem(w)
                            if stem not in seen_in_topic:
                                seen_in_topic.add(stem)
                                stem_topic_count[stem] = stem_topic_count.get(stem, 0) + 1
                threshold = max(2, len(real_topics) * 0.5)
                common_stems = {s for s, c in stem_topic_count.items() if c >= threshold}

            # Pick distinctive terms with bigram-aware dedup
            seen_stems, unique = set(), []
            for w in all_words:
                parts = w.replace('_', ' ').split()
                part_stems = [_word_stem(p) for p in parts]
                if any(s in common_stems for s in part_stems):
                    continue
                if any(s in seen_stems for s in part_stems):
                    continue
                for s in part_stems:
                    seen_stems.add(s)
                unique.append(w.replace('_', ' ').title())
                if len(unique) >= 3:
                    break

            # Fallback if filtering removed everything
            if not unique:
                seen_stems = set()
                for w in all_words:
                    parts = w.replace('_', ' ').split()
                    part_stems = [_word_stem(p) for p in parts]
                    if any(s in seen_stems for s in part_stems):
                        continue
                    for s in part_stems:
                        seen_stems.add(s)
                    unique.append(w.replace('_', ' ').title())
                    if len(unique) >= 3:
                        break

            if unique:
                if len(unique) <= 2:
                    return ', '.join(unique)
                return ', '.join(unique[:-1]) + ' & ' + unique[-1]

    # Fallback: parse from the name string
    name = re.sub(r'^\-?\d+_', '', str(name))
    words = [w.strip().replace('_', ' ').title() for w in name.split('_') if w.strip()]
    seen = set()
    unique = []
    for w in words:
        if w.lower() not in seen:
            seen.add(w.lower())
            unique.append(w)
        if len(unique) >= 3:
            break
    if len(unique) <= 2:
        return ', '.join(unique)
    return ', '.join(unique[:-1]) + ' & ' + unique[-1]


def on_run_clicked(_):
    global topic_model, topics, probs, docs_processed, docs_original, labels_valid
    run_out.clear_output()
    progress.value = ''

    if not documents:
        with run_out:
            print("\u26a0\ufe0f No documents loaded. Run the Data Input cell first.")
        return

    with run_out:
        print(f"Running BERTopic on {len(documents)} text segments...")

    progress.value = '<span style="color: #6096BA;">Preprocessing text...</span>'

    # Preprocess
    if preprocess_chk.value:
        docs_processed = [preprocess_text(d) for d in documents]
    else:
        docs_processed = [str(d) for d in documents]

    # Remove empty docs but track originals
    valid = [(d, l, orig) for d, l, orig in zip(docs_processed, doc_labels, documents) if d.strip()]
    if not valid:
        with run_out:
            print("\u26a0\ufe0f All segments were empty after preprocessing.")
        progress.value = ''
        return

    docs_clean, labels_valid_local, originals = zip(*valid)
    docs_clean = list(docs_clean)
    docs_original = list(originals)
    labels_valid = list(labels_valid_local)

    with run_out:
        print(f"   {len(docs_clean)} segments after preprocessing")

    progress.value = '<span style="color: #6096BA;">Building topic model...</span>'

    try:
        # Use academic stopwords in the vectorizer so topic labels are distinctive
        vectorizer = CountVectorizer(
            stop_words=list(ACADEMIC_STOPWORDS),
            ngram_range=(1, ngram_input.value),
            min_df=2,
        )
        representation_model = KeyBERTInspired()

        # HDBSCAN tuned for granular topic discovery
        from hdbscan import HDBSCAN
        n_docs = len(docs_clean)
        # Scale cluster size: smaller corpora need smaller clusters
        auto_cluster_size = max(3, min(min_topic_input.value, n_docs // 15))
        hdbscan_model = HDBSCAN(
            min_cluster_size=auto_cluster_size,
            min_samples=1,
            metric='euclidean',
            cluster_selection_method='leaf',  # 'leaf' produces more fine-grained clusters than 'eom'
            prediction_data=True,
        )

        model_kwargs = {
            'min_topic_size': min_topic_input.value,
            'hdbscan_model': hdbscan_model,
            'vectorizer_model': vectorizer,
            'representation_model': representation_model,
            'verbose': False,
        }

        # Zero-shot config
        if mode_toggle.value == 'zeroshot':
            zt = [t.strip() for t in zeroshot_input.value.strip().split('\n') if t.strip()]
            if zt:
                model_kwargs['zeroshot_topic_list'] = zt
                model_kwargs['zeroshot_min_similarity'] = zeroshot_sim.value
                with run_out:
                    print(f"   Zero-shot topics: {zt}")

        topic_model = BERTopic(**model_kwargs)

        # Suppress model loading noise (safetensors warnings bypass Python logging)
        import contextlib, io as _io, sys as _sys
        _stderr_backup = _sys.stderr
        _sys.stderr = _io.StringIO()
        try:
            topics, probs = topic_model.fit_transform(docs_clean)
        finally:
            _sys.stderr = _stderr_backup

        # Dynamic corpus stopwords: find terms appearing in >40% of segments
        # and re-fit the representation with them filtered out
        progress.value = '<span style="color: #6096BA;">Refining topic labels...</span>'

        try:
            from sklearn.feature_extraction.text import CountVectorizer as _CV
            _temp_vec = _CV(stop_words=list(ACADEMIC_STOPWORDS))
            _dtm = _temp_vec.fit_transform(docs_clean)
            _vocab = _temp_vec.get_feature_names_out()
            _doc_freq = (_dtm > 0).sum(axis=0).A1
            _threshold = len(docs_clean) * 0.4
            _corpus_stopwords = {_vocab[i] for i in range(len(_vocab)) if _doc_freq[i] >= _threshold}

            # Also filter citation noise: years, short tokens, pure numbers
            _vocab_idx = {v: i for i, v in enumerate(_vocab)}
            for w in list(_vocab):
                wl = w.lower().strip()
                # Years (1900-2099)
                if re.match(r'^(19|20)\d{2}$', wl):
                    _corpus_stopwords.add(wl)
                # Pure numbers or single/double chars
                if len(wl) <= 2 or wl.isdigit():
                    _corpus_stopwords.add(wl)
                # Common name-like tokens that appear in citations (detect by frequency + short length)
                if len(wl) <= 5 and _doc_freq[_vocab_idx[w]] > len(docs_clean) * 0.15:
                    _corpus_stopwords.add(wl)

            if _corpus_stopwords:
                _all_stops = list(ACADEMIC_STOPWORDS | _corpus_stopwords)
                _new_vectorizer = CountVectorizer(
                    stop_words=_all_stops,
                    ngram_range=(1, ngram_input.value),
                    min_df=2,
                )
                topic_model.update_topics(docs_clean, vectorizer_model=_new_vectorizer)
                topics = topic_model.topics_
        except Exception:
            pass  # Fall back to original representation if this fails

        # Reduce topics if requested
        nr = nr_topics_input.value
        if nr is not None and nr != 'auto':
            topic_model.reduce_topics(docs_clean, nr_topics=int(nr))
            topics = topic_model.topics_

        progress.value = ''

        with run_out:
            topic_info = topic_model.get_topic_info()
            real_topics = topic_info[topic_info['Topic'] != -1]
            n_topics = len(real_topics)
            n_outliers = len([t for t in topics if t == -1])

            # Header
            summary_html = '<div style="background-color: #E7ECEF; padding: 20px; border-radius: 10px; margin: 10px 0; border-left: 5px solid #274C77;">'
            summary_html += f'<h3 style="color: #274C77; margin-top: 0;">Topic Model Results</h3>'
            summary_html += f'<p><strong>{n_topics} topics</strong> discovered across {len(docs_clean)} text segments'
            if n_outliers:
                summary_html += f' ({n_outliers} outlier segments not assigned to any topic)'
            summary_html += '</p></div>'
            display(HTML(summary_html))

            # Per-topic cards
            for _, row in real_topics.iterrows():
                tid = row['Topic']
                raw_name = row['Name']
                count = row['Count']
                label = clean_topic_label(raw_name, topic_id=tid, topic_model_ref=topic_model)

                # Get keywords
                topic_words = topic_model.get_topic(tid)
                keywords = [w for w, _ in topic_words[:8]] if topic_words else []

                # Get representative original texts
                topic_indices = [i for i, t in enumerate(topics) if t == tid]
                rep_texts = [docs_original[i] for i in topic_indices[:3]]

                # Source file breakdown
                topic_labels = [labels_valid[i] for i in topic_indices]
                source_files = {}
                for lbl in topic_labels:
                    fname = lbl.rsplit('_', 1)[0]
                    source_files[fname] = source_files.get(fname, 0) + 1

                # Build card
                card = f'<div style="background: white; border: 1px solid #E7ECEF; border-radius: 10px; padding: 20px; margin: 15px 0; border-left: 4px solid #6096BA;">'
                card += f'<h4 style="color: #274C77; margin: 0 0 8px 0;">Topic {tid}: {label}</h4>'
                card += f'<p style="color: #8B8C89; margin: 0 0 12px 0;">{count} segments</p>'

                # Keywords as pills
                card += '<div style="margin-bottom: 12px;">'
                for kw in keywords:
                    card += f'<span style="background: #A3CEF1; color: #274C77; padding: 3px 10px; border-radius: 12px; font-size: 0.85em; margin: 2px 4px 2px 0; display: inline-block;">{kw}</span>'
                card += '</div>'

                # Source files
                if source_files:
                    card += '<p style="margin: 8px 0 4px 0; font-size: 0.85em; color: #274C77;"><strong>Sources:</strong></p>'
                    card += '<div style="margin-bottom: 10px;">'
                    for fname, cnt in sorted(source_files.items(), key=lambda x: x[1], reverse=True):
                        short = fname[:50] + '...' if len(fname) > 50 else fname
                        card += f'<span style="font-size: 0.85em; color: #8B8C89;">{short} ({cnt})</span><br>'
                    card += '</div>'

                # Representative passages
                card += '<p style="margin: 8px 0 4px 0; font-size: 0.85em; color: #274C77;"><strong>Representative passages:</strong></p>'
                for txt in rep_texts:
                    excerpt = txt[:300] + '...' if len(txt) > 300 else txt
                    card += f'<div style="background: #F8F9FA; padding: 10px; border-radius: 6px; margin: 6px 0; font-size: 0.85em; line-height: 1.5; color: #333;">{excerpt}</div>'

                card += '</div>'
                display(HTML(card))

            # --- Source-by-Topic Heatmap ---
            # Build a matrix of how many segments each source file contributes to each topic
            source_topic_data = {}
            for i_idx, t in enumerate(topics):
                if t == -1:
                    continue
                if i_idx < len(labels_valid):
                    fname = labels_valid[i_idx].rsplit('_', 1)[0]
                    short = fname[:45] + '...' if len(fname) > 45 else fname
                    if short not in source_topic_data:
                        source_topic_data[short] = {}
                    topic_label = clean_topic_label(
                        real_topics[real_topics['Topic'] == t].iloc[0]['Name'] if t in real_topics['Topic'].values else str(t),
                        topic_id=t, topic_model_ref=topic_model)
                    col_name = f'T{t}: {topic_label}'
                    source_topic_data[short][col_name] = source_topic_data[short].get(col_name, 0) + 1

            if source_topic_data:
                heatmap_df = pd.DataFrame(source_topic_data).T.fillna(0).astype(int)
                # Sort columns by topic ID, rows by total segments
                heatmap_df['_total'] = heatmap_df.sum(axis=1)
                heatmap_df = heatmap_df.sort_values('_total', ascending=False).drop('_total', axis=1)

                hm_html = '<div style="margin-top: 25px;">'
                hm_html += '<h4 style="color: #274C77;">Source File by Topic Distribution</h4>'
                hm_html += '<p style="color: #8B8C89; font-size: 0.85em;">Shows how segments from each file are distributed across topics. Helps identify which documents drive which themes.</p>'
                hm_html += '<table style="border-collapse: collapse; font-size: 0.85em; margin-top: 10px;">'

                # Header row
                hm_html += '<tr><th style="padding: 8px 12px; border: 1px solid #E7ECEF; background: #274C77; color: white; text-align: left;">Source File</th>'
                for col in heatmap_df.columns:
                    hm_html += f'<th style="padding: 8px 10px; border: 1px solid #E7ECEF; background: #274C77; color: white; text-align: center; max-width: 150px;">{col}</th>'
                hm_html += '</tr>'

                # Data rows with background color intensity
                max_val = heatmap_df.values.max() if heatmap_df.values.max() > 0 else 1
                for fname, row_data in heatmap_df.iterrows():
                    hm_html += f'<tr><td style="padding: 6px 12px; border: 1px solid #E7ECEF; font-weight: bold; white-space: nowrap;">{fname}</td>'
                    for val in row_data:
                        intensity = val / max_val
                        if val == 0:
                            bg = '#FAFAFA'
                            color = '#CCC'
                        else:
                            # Gradient from light blue to dark blue
                            r = int(231 - (231 - 39) * intensity)
                            g = int(236 - (236 - 76) * intensity)
                            b = int(239 - (239 - 119) * intensity)
                            bg = f'rgb({r},{g},{b})'
                            color = '#FFF' if intensity > 0.5 else '#274C77'
                        hm_html += f'<td style="padding: 6px 10px; border: 1px solid #E7ECEF; text-align: center; background: {bg}; color: {color}; font-weight: {"bold" if intensity > 0.3 else "normal"};">{int(val)}</td>'
                    hm_html += '</tr>'

                hm_html += '</table></div>'
                display(HTML(hm_html))

            # Outliers note
            if n_outliers:
                out_html = f'<p style="color: #8B8C89; font-style: italic; margin-top: 10px;">{n_outliers} segments were not assigned to any topic (outliers). These are segments that did not cluster strongly with others.</p>'
                display(HTML(out_html))

    except Exception as e:
        progress.value = ''
        with run_out:
            import traceback
            print(f"\u274c Error: {type(e).__name__}: {e}")
            traceback.print_exc()

run_btn.on_click(on_run_clicked)
display(widgets.VBox([run_btn, progress, run_out]))

## Visualizations

Explore the discovered topics through interactive visualizations. Each visualization offers a different perspective on the topic structure.

In [ ]:
# Topic Visualizations and Browser

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    topic_info = topic_model.get_topic_info()
    real_topics = topic_info[topic_info['Topic'] != -1].sort_values('Count', ascending=False)

    # --- 1. Top Terms per Topic (horizontal bar charts) ---
    print("\U0001f4ca Top Terms per Topic")
    print()

    n_topics_to_show = min(10, len(real_topics))
    fig, axes = plt.subplots(n_topics_to_show, 1, figsize=(10, n_topics_to_show * 2.2))
    if n_topics_to_show == 1:
        axes = [axes]

    for ax_idx, (_, row) in enumerate(real_topics.head(n_topics_to_show).iterrows()):
        tid = row['Topic']
        label = clean_topic_label(row['Name'], topic_id=tid, topic_model_ref=topic_model)
        topic_words = topic_model.get_topic(tid)
        if topic_words:
            words = [w for w, _ in topic_words[:8]]
            scores = [s for _, s in topic_words[:8]]
            words.reverse()
            scores.reverse()

            ax = axes[ax_idx]
            bars = ax.barh(words, scores, color='#6096BA', edgecolor='#274C77', linewidth=0.5)
            ax.set_title(f'Topic {tid}: {label} ({row["Count"]} segs)', fontsize=11, color='#274C77', fontweight='bold', loc='left')
            ax.set_xlim(0, max(scores) * 1.1 if scores else 1)
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_color('#E7ECEF')
            ax.spines['bottom'].set_color('#E7ECEF')
            ax.tick_params(colors='#8B8C89', labelsize=9)

    plt.tight_layout(pad=1.5)
    plt.show()

    # --- 2. Topic Size Distribution ---
    print()
    print("\U0001f4ca Topic Size Distribution")
    print()

    fig, ax = plt.subplots(figsize=(10, 4))
    topic_labels_list = []
    topic_counts = []
    for _, row in real_topics.iterrows():
        tid = row['Topic']
        label = clean_topic_label(row['Name'], topic_id=tid, topic_model_ref=topic_model)
        short_label = label[:30] + '...' if len(label) > 30 else label
        topic_labels_list.append(f'T{tid}: {short_label}')
        topic_counts.append(row['Count'])

    bars = ax.barh(topic_labels_list[::-1], topic_counts[::-1], color='#6096BA', edgecolor='#274C77', linewidth=0.5)
    ax.set_xlabel('Number of Segments', fontsize=10, color='#274C77')
    ax.set_title('Segments per Topic', fontsize=13, color='#274C77', fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#E7ECEF')
    ax.spines['bottom'].set_color('#E7ECEF')
    ax.tick_params(colors='#8B8C89', labelsize=9)

    # Add count labels on bars
    for bar, count in zip(bars, topic_counts[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(count),
                va='center', fontsize=9, color='#274C77')

    plt.tight_layout()
    plt.show()

    # --- 3. Source-by-Topic Heatmap ---
    print()
    print("\U0001f4ca Source File by Topic Distribution")
    print()

    source_topic_data = {}
    for i_idx, t in enumerate(topics):
        if t == -1:
            continue
        if i_idx < len(labels_valid):
            fname = labels_valid[i_idx].rsplit('_', 1)[0]
            short = fname[:40] + '...' if len(fname) > 40 else fname
            if short not in source_topic_data:
                source_topic_data[short] = {}
            tl = clean_topic_label(
                real_topics[real_topics['Topic'] == t].iloc[0]['Name'] if t in real_topics['Topic'].values else str(t),
                topic_id=t, topic_model_ref=topic_model)
            col = f'T{t}'
            source_topic_data[short][col] = source_topic_data[short].get(col, 0) + 1

    if source_topic_data:
        heatmap_df = pd.DataFrame(source_topic_data).T.fillna(0).astype(int)
        # Sort columns by topic ID number
        heatmap_df = heatmap_df[sorted(heatmap_df.columns, key=lambda x: int(x.replace('T', '')))]
        # Sort rows by total segments
        heatmap_df['_total'] = heatmap_df.sum(axis=1)
        heatmap_df = heatmap_df.sort_values('_total', ascending=False).drop('_total', axis=1)

        fig, ax = plt.subplots(figsize=(max(8, len(heatmap_df.columns) * 1.2), max(4, len(heatmap_df) * 0.6)))
        im = ax.imshow(heatmap_df.values, cmap='Blues', aspect='auto')

        ax.set_xticks(range(len(heatmap_df.columns)))
        ax.set_xticklabels(heatmap_df.columns, fontsize=9, color='#274C77', rotation=45, ha='right')
        ax.set_yticks(range(len(heatmap_df.index)))
        ax.set_yticklabels(heatmap_df.index, fontsize=9, color='#274C77')

        # Add value labels
        for i in range(len(heatmap_df.index)):
            for j in range(len(heatmap_df.columns)):
                val = heatmap_df.values[i, j]
                if val > 0:
                    color = 'white' if val > heatmap_df.values.max() * 0.5 else '#274C77'
                    ax.text(j, i, str(int(val)), ha='center', va='center', fontsize=9, color=color, fontweight='bold')

        ax.set_title('Segments per Source File and Topic', fontsize=13, color='#274C77', fontweight='bold', pad=15)
        plt.colorbar(im, ax=ax, shrink=0.8, label='Segment count')
        plt.tight_layout()
        plt.show()

    # --- Topic Browser ---
    browse_header = widgets.HTML('<h3 style="margin: 30px 0 10px 0; color: #274C77;">Browse Topic Segments</h3>')
    browse_help = widgets.HTML('<p style="color: #8B8C89; font-size: 0.85em;">Select a topic to see all text segments assigned to it.</p>')

    sorted_ti = pd.concat([real_topics, topic_info[topic_info['Topic'] == -1]])

    topic_options = []
    for _, row in sorted_ti.iterrows():
        tid = row['Topic']
        label = clean_topic_label(row['Name'], topic_id=tid, topic_model_ref=topic_model)
        topic_options.append((f"Topic {tid}: {label} ({row['Count']} segments)", tid))

    browse_dropdown = widgets.Dropdown(
        options=topic_options,
        description='Topic:',
        style={'description_width': '60px'},
        layout=widgets.Layout(width='600px')
    )

    browse_out = widgets.Output()

    def show_topic_segments(change=None):
        browse_out.clear_output()
        tid = browse_dropdown.value

        with browse_out:
            indices = [i for i, t in enumerate(topics) if t == tid]
            if not indices:
                print("No segments assigned to this topic.")
                return

            label = clean_topic_label(topic_info[topic_info['Topic'] == tid].iloc[0]['Name'], topic_id=tid, topic_model_ref=topic_model)
            header = f'<div style="background-color: #E7ECEF; padding: 12px; border-radius: 8px; margin-bottom: 10px;">'
            header += f'<strong style="color: #274C77;">Topic {tid}: {label}</strong> \u2014 {len(indices)} segments</div>'
            display(HTML(header))

            for rank, idx in enumerate(indices, 1):
                orig = docs_original[idx]
                source = labels_valid[idx].rsplit('_', 1)[0]
                seg_html = f'<div style="border-left: 3px solid #6096BA; padding: 8px 12px; margin: 8px 0; background: #FAFAFA; border-radius: 4px;">'
                seg_html += f'<span style="color: #8B8C89; font-size: 0.8em;">{rank}. {source}</span><br>'
                seg_html += f'<span style="font-size: 0.9em; line-height: 1.5;">{orig[:500]}{"..." if len(orig) > 500 else ""}</span>'
                seg_html += '</div>'
                display(HTML(seg_html))

    browse_dropdown.observe(show_topic_segments, names='value')

    display(widgets.VBox([browse_header, browse_help, browse_dropdown, browse_out]))
    show_topic_segments()

## Topic Search

Search for topics similar to any keyword or phrase. Useful for finding which discovered topics relate to concepts you care about.

In [ ]:
# Topic Search

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    search_input = widgets.Text(value='', placeholder='Enter a keyword or phrase', description='Search:', layout=layout, style=style)
    search_btn = widgets.Button(description='Find Topics', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='150px', margin='10px 0'))
    search_out = widgets.Output()

    def on_search(_):
        search_out.clear_output()
        query = search_input.value.strip()
        if not query:
            return
        with search_out:
            similar_topics, similarity = topic_model.find_topics(query, top_n=5)
            print(f"\U0001f50d Topics similar to \"{query}\":\n")
            ti = topic_model.get_topic_info()
            for t, s in zip(similar_topics, similarity):
                raw_name = ti[ti['Topic'] == t].iloc[0]['Name'] if t in ti['Topic'].values else str(t)
                label = clean_topic_label(raw_name, topic_id=t, topic_model_ref=topic_model)
                topic_words = topic_model.get_topic(t)
                kw = ', '.join([w for w, _ in topic_words[:5]]) if topic_words else ''
                print(f"  Topic {t}: {label} (similarity: {s:.3f})")
                print(f"    Keywords: {kw}")

    search_btn.on_click(on_search)
    display(widgets.VBox([search_input, search_btn, search_out]))

## Export Results

Export topic assignments and summaries as CSV or Excel. The export includes document-level topic assignments and a topic summary sheet.

In [ ]:
# Export Results

if topic_model is None:
    print("\u26a0\ufe0f Run the topic model first.")
else:
    export_format = widgets.Dropdown(options=[('Excel (.xlsx) \u2014 full report', 'xlsx'), ('CSV (.csv)', 'csv')], value='xlsx', description='Format:', style=style, layout=layout)
    export_btn = widgets.Button(description='Export Results', style={'button_color': '#6096BA'}, layout=widgets.Layout(width='200px', margin='10px 0'))
    export_out = widgets.Output()

    def on_export(_):
        export_out.clear_output()
        with export_out:
            try:
                timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                base = f'bertopic_results_{timestamp}'

                topic_info = topic_model.get_topic_info()

                # Document-level assignments with original text and readable labels
                doc_rows = []
                for i in range(len(topics)):
                    tid = topics[i]
                    raw_name = topic_info[topic_info['Topic'] == tid].iloc[0]['Name'] if tid in topic_info['Topic'].values else 'Outlier'
                    doc_rows.append({
                        'segment': docs_original[i][:500] if i < len(docs_original) else '',
                        'source_file': labels_valid[i].rsplit('_', 1)[0] if i < len(labels_valid) else '',
                        'topic_id': tid,
                        'topic_label': clean_topic_label(raw_name, topic_id=tid, topic_model_ref=topic_model),
                    })
                doc_df = pd.DataFrame(doc_rows)

                # Topic summary with keywords and representative passages
                summary_rows = []
                for _, row in topic_info.iterrows():
                    tid = row['Topic']
                    label = clean_topic_label(row['Name'], topic_id=row['Topic'], topic_model_ref=topic_model)
                    topic_words = topic_model.get_topic(tid)
                    keywords = ', '.join([w for w, _ in topic_words[:10]]) if topic_words else ''

                    # Representative passages
                    indices = [i for i, t in enumerate(topics) if t == tid]
                    reps = [docs_original[i][:300] for i in indices[:3]] if docs_original else []

                    # Source breakdown
                    sources = {}
                    for idx in indices:
                        if idx < len(labels_valid):
                            fname = labels_valid[idx].rsplit('_', 1)[0]
                            sources[fname] = sources.get(fname, 0) + 1
                    source_str = '; '.join(f'{k} ({v})' for k, v in sorted(sources.items(), key=lambda x: x[1], reverse=True))

                    summary_rows.append({
                        'topic_id': tid,
                        'topic_label': label,
                        'segment_count': row['Count'],
                        'keywords': keywords,
                        'source_files': source_str,
                        'representative_1': reps[0] if len(reps) > 0 else '',
                        'representative_2': reps[1] if len(reps) > 1 else '',
                        'representative_3': reps[2] if len(reps) > 2 else '',
                    })
                summary_df = pd.DataFrame(summary_rows)

                fmt = export_format.value
                if fmt == 'xlsx':
                    filepath = f'{base}.xlsx'
                    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
                        summary_df.to_excel(writer, sheet_name='Topic Summary', index=False)
                        doc_df.to_excel(writer, sheet_name='Segment Assignments', index=False)
                    style_excel(filepath)
                else:
                    filepath = f'{base}_segments.csv'
                    doc_df.to_csv(filepath, index=False)
                    summary_path = f'{base}_topics.csv'
                    summary_df.to_csv(summary_path, index=False)

                print(f"\u2713 Saved: {filepath}")
                print(f"   {len(summary_df)} topics, {len(doc_df)} segments")

                if IN_COLAB:
                    colab_files.download(filepath)
                    if fmt == 'csv':
                        colab_files.download(summary_path)

            except Exception as e:
                print(f"\u274c Export error: {type(e).__name__}: {e}")

    export_btn.on_click(on_export)
    display(widgets.VBox([export_format, export_btn, export_out]))